In [0]:
# # ⚠️ WARNING: This cell deletes the checkpoint and reprocesses ALL files
# # USE ONLY for debugging/recovery - DO NOT include in scheduled jobs!
# # Reset the checkpoint to force reprocessing all files
# import shutil
# import os

# checkpoint_path = "/Volumes/retail_p/volumes/blob_source/_checkpoint/transactions"

# # Remove checkpoint directory
# dbutils.fs.rm(checkpoint_path, recurse=True)
# print(f"Checkpoint reset: {checkpoint_path}")

# # Now re-run Cell 1 to reload all the data

In [0]:
# ============================================================
# PYTHON AUTO LOADER - Works on Serverless compute
# ============================================================

# Auto Loader to read CSV files and write to bronze table
from pyspark.sql.functions import current_timestamp, col

# Source and target paths
source_path = "/Volumes/retail_p/volumes/blob_source/transaction_source/"
target_table = "retail_p.blob_bonze.transactions"

# Read CSV files using Auto Loader (cloudFiles)
df = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .option("cloudFiles.schemaLocation", "/Volumes/retail_p/volumes/blob_source/_schema_location/transactions")
  .load(source_path)
  .withColumn("ingestion_timestamp", current_timestamp())
  .withColumn("source_file", col("_metadata.file_path"))
)

# Write to bronze table using Auto Loader
# Using trigger(once=True) to process available files once and stop
query = (df.writeStream
  .format("delta")
  .option("checkpointLocation", "/Volumes/retail_p/volumes/blob_source/_checkpoint/transactions")
  .option("mergeSchema", "true")
  .trigger(availableNow=True)
  .toTable(target_table)
)

# Wait for the stream to complete and stop
query.awaitTermination()
print(f"Successfully loaded data to {target_table}")

In [0]:
%sql
SELECT COUNT(*) FROM retail_p.blob_bonze.transactions;

In [0]:
# %sql
# DROP TABLE retail_p.blob_bonze.transactions;

In [0]:
%sql
-- ============================================================
-- SQL STREAMING TABLE VERSION
-- ⚠️ NOT SUPPORTED on Serverless compute (requires regular cluster or Lakeflow Pipeline)
-- ============================================================

-- This SQL approach is simpler but only works on:
-- 1. Regular clusters (all-purpose or job clusters)
-- 2. Lakeflow Spark Declarative Pipelines
-- 3. Serverless with preview feature enabled (contact Databricks)

-- CREATE OR REFRESH STREAMING TABLE retail_p.blob_bonze.transactions
-- COMMENT 'Bronze table for transaction data from blob storage'
-- AS
-- SELECT 
--   *,
--   current_timestamp() as ingestion_timestamp,
--   _metadata.file_path as source_file
-- FROM read_files(
--   '/Volumes/retail_p/volumes/blob_source/transaction_source/',
--   format => 'csv',
--   header => true
-- )